# KL divergence — companion notebook

Runnable companion for [`kl-divergence.md`](./kl-divergence.md).

1. Asymmetry: $D(P \Vert Q) \neq D(Q \Vert P)$ numerically (Bernoulli, Beta, Gaussian).
2. Closed-form Gaussian–Gaussian KL vs Monte-Carlo estimate.
3. Forward vs reverse KL projection of a bimodal mixture onto a single Gaussian (mean-seeking vs mode-seeking).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import xlogy

rng = np.random.default_rng(0)

def kl_discrete(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    return float(np.sum(xlogy(p, p) - xlogy(p, q)))

def kl_density(p_vals, q_vals, dx):
    return float(np.sum(xlogy(p_vals, p_vals) - xlogy(p_vals, q_vals)) * dx)

## 1. Asymmetry on Bernoullis and Betas

In [ ]:
P = np.array([0.9, 0.1])  # Ber(0.9)
Q = np.array([0.5, 0.5])  # Ber(0.5)
print(f'Bernoulli: D(P||Q) = {kl_discrete(P, Q):.4f} nats')
print(f'           D(Q||P) = {kl_discrete(Q, P):.4f} nats')

# Beta-Beta via grid quadrature
xs = np.linspace(1e-4, 1 - 1e-4, 4000)
dx = xs[1] - xs[0]
p_beta = stats.beta.pdf(xs, 2, 5)
q_beta = stats.beta.pdf(xs, 5, 2)
print(f'\nBeta(2,5) vs Beta(5,2):')
print(f'  D(P||Q) = {kl_density(p_beta, q_beta, dx):.4f}')
print(f'  D(Q||P) = {kl_density(q_beta, p_beta, dx):.4f}')

## 2. Gaussian–Gaussian KL: closed form vs Monte Carlo

In [ ]:
def kl_gauss(mu1, s1, mu2, s2):
    return np.log(s2 / s1) + (s1**2 + (mu1 - mu2)**2) / (2 * s2**2) - 0.5

mu1, s1 = 0.0, 1.0
mu2, s2 = 1.0, 2.0
closed = kl_gauss(mu1, s1, mu2, s2)

# Monte Carlo: E_P[log p - log q]
N = 200_000
x = rng.normal(mu1, s1, N)
logp = stats.norm.logpdf(x, mu1, s1)
logq = stats.norm.logpdf(x, mu2, s2)
mc = float(np.mean(logp - logq))
se = float(np.std(logp - logq, ddof=1) / np.sqrt(N))

print(f'closed form D(N({mu1},{s1}^2) || N({mu2},{s2}^2)) = {closed:.6f}')
print(f'monte carlo                                       = {mc:.6f}  (+/- {1.96*se:.6f})')

## 3. Forward vs reverse KL projection

Target $P$ is a bimodal mixture; fit a single Gaussian $Q = \mathcal{N}(\mu, \sigma^2)$ by minimizing

- **Forward** $D(P \Vert Q)$: minimized (analytically) by moment matching $\mu = \mathbb{E}_P[X]$, $\sigma^2 = \operatorname{Var}_P(X)$. Mean-seeking — covers both modes.
- **Reverse** $D(Q \Vert P)$: typically locks onto a single mode. Mode-seeking.

We solve forward analytically and reverse by grid search over $(\mu, \sigma)$.

In [ ]:
# Mixture P = 0.5 N(-3, 1^2) + 0.5 N(3, 1^2)
w = np.array([0.5, 0.5])
mus = np.array([-3.0, 3.0])
sds = np.array([1.0, 1.0])

def p_pdf(x):
    return w[0] * stats.norm.pdf(x, mus[0], sds[0]) + w[1] * stats.norm.pdf(x, mus[1], sds[1])

xs = np.linspace(-8, 8, 4000)
dx = xs[1] - xs[0]
P_vals = p_pdf(xs)

# Forward: moment matching
mu_P  = float(np.sum(xs * P_vals) * dx)
var_P = float(np.sum((xs - mu_P)**2 * P_vals) * dx)
mu_fwd, sd_fwd = mu_P, np.sqrt(var_P)

# Reverse: grid search minimizing D(Q || P) = int q log(q/p)
mu_grid = np.linspace(-5, 5, 81)
sd_grid = np.linspace(0.3, 3.5, 81)
best = (np.inf, None, None)
for mu in mu_grid:
    for sd in sd_grid:
        Q_vals = stats.norm.pdf(xs, mu, sd)
        # integrand q (log q - log p); xlogy handles q=0
        integrand = xlogy(Q_vals, Q_vals) - xlogy(Q_vals, P_vals)
        d = float(np.sum(integrand) * dx)
        if d < best[0]:
            best = (d, mu, sd)
_, mu_rev, sd_rev = best

print(f'Forward KL minimizer (moment match): mu = {mu_fwd:.3f}, sigma = {sd_fwd:.3f}')
print(f'Reverse KL minimizer (grid search): mu = {mu_rev:.3f}, sigma = {sd_rev:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xs, P_vals, 'k', lw=2, label='target P (mixture)')
ax.plot(xs, stats.norm.pdf(xs, mu_fwd, sd_fwd), 'tab:blue', lw=2,
        label=f'forward KL fit  (mean-seeking)')
ax.plot(xs, stats.norm.pdf(xs, mu_rev, sd_rev), 'tab:red', lw=2,
        label=f'reverse KL fit  (mode-seeking)')
ax.set_xlabel('x'); ax.set_ylabel('density')
ax.set_title('Forward vs reverse KL: same family, different minimizer')
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()